# Pipeline Health Check
End-to-end checklist for the current KKBox churn pipeline.

Run cells top-to-bottom to verify the current state of:
- feature engineering
- preprocessing and split artifacts
- model training outputs
- log/output locations

In [ ]:
# Setup
import sys
from pathlib import Path

import yaml
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'config').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'config').exists():
    raise RuntimeError('Could not locate project root')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / 'config' / 'config.yaml'
with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    config = yaml.safe_load(handle)

TARGET = config['project']['target_col']
CUTOFF = config['feature_engineering']['cutoff_date']
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim'
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS_DIR = PROJECT_ROOT / 'reports'
LOGS_DIR = PROJECT_ROOT / 'logs'

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'CUTOFF     : {CUTOFF}')
print(f'TARGET     : {TARGET}')
print(f'Candidates : {config["modeling"]["candidate_models"]}')

In [ ]:
# Artifact inventory
paths = [
    ('data/interim', INTERIM_DIR),
    ('data/processed', PROCESSED_DIR),
    ('models', MODELS_DIR),
    ('reports', REPORTS_DIR),
    ('logs', LOGS_DIR),
]

for label, folder in paths:
    print(f'\n[{label}]')
    if not folder.exists():
        print('  MISSING')
        continue
    items = sorted(folder.iterdir())
    if not items:
        print('  (empty)')
        continue
    for item in items:
        size_mb = item.stat().st_size / 1024 / 1024
        suffix = '/' if item.is_dir() else ''
        print(f'  {item.name}{suffix:<2}  {size_mb:.2f} MB')

In [ ]:
# Feature engineering sanity checks
feature_path = PROCESSED_DIR / config['feature_engineering']['output_file']
if not feature_path.exists():
    raise FileNotFoundError(f'Missing feature frame: {feature_path}')

ff = pd.read_parquet(feature_path)
print(f'Feature frame shape: {ff.shape}')
print(f'analysis_reference_date: {ff["analysis_reference_date"].iloc[0] if "analysis_reference_date" in ff.columns else "MISSING"}')

for col in ['last_transaction_date', 'last_log_date', 'last_expire_date']:
    if col in ff.columns:
        print(f'{col}: min={ff[col].min()}  max={ff[col].max()}')

null_counts = ff.isna().sum().sort_values(ascending=False)
print('\nTop columns with nulls:')
display(null_counts[null_counts > 0].head(15))

In [ ]:
# Split sanity checks
def _load_split(name: str):
    path = PROCESSED_DIR / f'{name}.parquet'
    if not path.exists():
        raise FileNotFoundError(f'Missing split file: {path}')
    return pd.read_parquet(path)

X_train = _load_split('X_train')
X_val = _load_split('X_val')
X_test = _load_split('X_test')
y_train = _load_split('y_train').iloc[:, 0]
y_val = _load_split('y_val').iloc[:, 0]
y_test = _load_split('y_test').iloc[:, 0]

print(f'X_train: {X_train.shape}')
print(f'X_val  : {X_val.shape}')
print(f'X_test : {X_test.shape}')
print()
print(f'churn rate train: {y_train.mean():.4f}')
print(f'churn rate val  : {y_val.mean():.4f}')
print(f'churn rate test : {y_test.mean():.4f}')

overlap_tv = len(set(X_train.index) & set(X_val.index))
overlap_tt = len(set(X_train.index) & set(X_test.index))
overlap_vt = len(set(X_val.index) & set(X_test.index))
print()
print(f'Index overlaps | train/val={overlap_tv}  train/test={overlap_tt}  val/test={overlap_vt}')

In [ ]:
# Training summary
comparison_path = REPORTS_DIR / 'model_comparison.csv'
if comparison_path.exists():
    comp = pd.read_csv(comparison_path)
    display(comp)
    champion = comp.sort_values('val_auc_pr', ascending=False).iloc[0]
    print()
    print(f'Champion: {champion["model"]}')
    print(f'  val_auc_pr : {champion["val_auc_pr"]:.4f}')
    print(f'  val_auc_roc: {champion["val_auc_roc"]:.4f}')
else:
    print(f'Missing comparison file: {comparison_path}')

In [ ]:
# Log locations and saved artifacts
log_files = [
    LOGS_DIR / 'feature_engineering.log',
    LOGS_DIR / 'training.log',
]
for path in log_files:
    print(f'{path.name}: {"exists" if path.exists() else "missing"}')

print('\nSaved models:')
for name in ['champion_model.pkl', 'champion_name.txt', 'logistic_regression.pkl', 'random_forest.pkl', 'xgboost.pkl', 'lightgbm.pkl', 'preprocessor.pkl']:
    path = MODELS_DIR / name
    print(f'  {name}: {"exists" if path.exists() else "missing"}')

## Notes
- This notebook is for verification and reporting only.
- It does not retrain models unless you explicitly add a training cell.
- If any artifact is missing, run the corresponding script first: ingestion -> engineer -> preprocessing -> training.